# Loss Functions and Optimizers

Loss tells the model **how wrong** it is. The optimizer decides **how to fix** it.
Picking the wrong one is one of the most common DL mistakes.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## Loss Functions

| Task | Loss | PyTorch class |
|---|---|---|
| Regression | MSE | `nn.MSELoss()` |
| Binary classification | BCE | `nn.BCEWithLogitsLoss()` |
| Multi-class classification | Cross-entropy | `nn.CrossEntropyLoss()` |

### MSE — Mean Squared Error (regression)

In [ ]:
mse = nn.MSELoss()

preds  = torch.tensor([2.5, 0.0, 2.0, 8.0])
target = torch.tensor([3.0, -0.5, 2.0, 7.0])

print(mse(preds, target))   # mean of (pred - target)^2

# Sensitive to outliers — one large error dominates
preds_outlier = torch.tensor([2.5, 0.0, 2.0, 100.0])
print(mse(preds_outlier, target))   # much larger

### BCE — Binary Cross-Entropy (binary classification)

Use `BCEWithLogitsLoss` over `BCELoss` — it takes raw logits (pre-sigmoid), which is more numerically stable.

In [ ]:
bce = nn.BCEWithLogitsLoss()

logits = torch.tensor([2.0, -1.5, 0.5, -3.0])  # raw model output, NOT probabilities
labels = torch.tensor([1.0,  0.0, 1.0,  0.0])  # binary

print(bce(logits, labels))

# Convert logits → probabilities manually
probs = torch.sigmoid(logits)
print(probs)   # values in [0, 1]

### CrossEntropyLoss — Multi-class classification

Expects raw logits of shape `(N, C)` and integer class labels of shape `(N,)`. Applies softmax internally.

In [ ]:
ce = nn.CrossEntropyLoss()

# 4 samples, 3 classes
logits = torch.tensor([[2.0, 1.0, 0.1],
                        [0.1, 3.0, 0.2],
                        [1.5, 0.3, 2.1],
                        [0.0, 0.0, 5.0]])
labels = torch.tensor([0, 1, 2, 2])  # integer class indices

print(ce(logits, labels))

# Probabilities via softmax
probs = torch.softmax(logits, dim=1)
print(probs)   # each row sums to 1
print(probs.argmax(dim=1))   # predicted classes: [0, 1, 2, 2]

## Optimizers

All optimizers implement: `gradient → weight update`.

| Optimizer | Key idea | When to use |
|---|---|---|
| SGD | plain gradient step | convex problems, careful tuning |
| SGD + momentum | smooth noisy gradients | image models with schedules |
| Adam | adaptive per-param LR | default for most DL |
| AdamW | Adam + proper weight decay | transformers, fine-tuning |

In [ ]:
model = nn.Linear(10, 1)

# SGD — fixed learning rate
sgd = torch.optim.SGD(model.parameters(), lr=0.01)

# SGD + momentum — accumulates velocity, escapes shallow minima faster
sgd_m = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam — adapts lr per parameter using gradient history
adam = torch.optim.Adam(model.parameters(), lr=1e-3)

# AdamW — Adam with decoupled weight decay (preferred over Adam for most tasks)
adamw = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

## Visualize: SGD vs Adam convergence speed

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_regression

X_np, y_np = make_regression(n_samples=500, n_features=10, noise=20, random_state=42)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

def run(opt_cls, **kwargs):
    m = nn.Linear(10, 1)
    opt = opt_cls(m.parameters(), **kwargs)
    losses = []
    for _ in range(30):
        epoch_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = nn.MSELoss()(m(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
    return losses

sgd_losses  = run(torch.optim.SGD,  lr=0.01)
adam_losses = run(torch.optim.Adam, lr=1e-3)

plt.plot(sgd_losses,  label='SGD')
plt.plot(adam_losses, label='Adam')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('SGD vs Adam on regression')
plt.show()

## Learning Rate Schedulers

LR too high → oscillates, never converges.  
LR too low → converges, but slowly.  
Scheduler → start high, reduce over time.

In [ ]:
model = nn.Linear(10, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# StepLR: multiply lr by gamma every step_size epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

lrs = []
for epoch in range(20):
    # ... train one epoch ...
    lrs.append(optimizer.param_groups[0]['lr'])
    scheduler.step()   # call AFTER optimizer.step()

plt.plot(lrs)
plt.xlabel('Epoch')
plt.ylabel('LR')
plt.title('StepLR decay')
plt.show()

In [ ]:
# CosineAnnealingLR — smooth decay, popular in practice
optimizer2 = torch.optim.Adam(model.parameters(), lr=0.1)
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=20)

lrs2 = []
for _ in range(20):
    lrs2.append(optimizer2.param_groups[0]['lr'])
    sched2.step()

plt.plot(lrs2)
plt.xlabel('Epoch')
plt.ylabel('LR')
plt.title('CosineAnnealingLR')
plt.show()